# 📋 Gradient boosted regression trees
In this notebook, you will implement another ensemble method that combines multiple decision trees. This is called **gradient boosted trees**. You will also practice hyperparameter tuning for a model using GridSearchCV. 



In [1]:
# Import needed libraries
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

## ℹ️ Ensemble Trees at a Glance


|                | **Bagging / Random Forest** | **Gradient Boosting** |
|----------------|-----------------------------|-----------------------|
| **How trees are grown** | Independently, in parallel, on boot-strapped data | Sequentially, each on the current residuals |
| **How outputs are combined** | Average (regression) or vote (classification) | Sum of weak learners; each tree *corrects* the last |
| **Main effect** | Reduces **variance** | Reduces **bias** (iteratively refines fit) |
| **Parallel-friendly?** | Yes | No (depends on previous tree) |


All three are tree-based ensemble methods: **Bagging** and **Random Forest** build many *independent* trees in parallel and aggregate their votes, while Gradient Boosting *adds trees one-by-one*, each new tree fixing the mistakes made so far.

---
 

### 📊 Breast Cancer Wisconsin dataset
Here, we will use the same dataset we used before, the [Breaast cancer dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) ( also available [here](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) )

The cell below loads the data and splits it into the training and test sets (see notebook for the second week).

In [2]:
# see above: from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True)

# Split the data into training and testing sets
# Set seed for reproducibility
r_state = 0 # random_state
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.3,
                                                    stratify=y,
                                                    random_state=r_state)

### 📝Exercise 1: Implementing Gradient boosting classifier

#### Part 1: Initialize a Gradient boosting classifier 

Initialize a Gradient boosting classifier with 50 decision stumps as weak learners and `learning_rate`=$0.1$.  

> Documentation for [`GradientBoostingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html)

In [3]:
# see above: from sklearn.ensemble import GradientBoostingClassifier

# Make an instance of the Gradient boosting classifier
gbc = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1,
                                random_state=0)

#### Part 2: Train the Gradient boosting classifier

Now train this model object with the training data we created earlier: `X_train` and `y_train`.

In [4]:
# Train the model on the data (train set)
gbc.fit(X_train, y_train)

GradientBoostingClassifier(n_estimators=50, random_state=0)

#### Part 3: Predict with Gradient boosting classifier

1. Generate predictions on `X_test` using the fitted `GradientBoostingClassifier` object
2. Save the results in variable `test_prediction`. 

In [5]:
# Predict the labels of new data (test set)

test_prediction = gbc.predict(X_test)  

#### Part 4: Evaluate the Gradient boosting classifier

- Use a scikit-learn function to compute the accuracy of this model on the test set.

In [6]:
test_accuracy = accuracy_score(y_test, test_prediction)

print('Accuracy of Gradient boosting classifier on test set: ', round(test_accuracy, 3))

Accuracy of Gradient boosting classifier on test set:  0.936


#### Part 5: Compair the performance

- Compare the performance of the Gradient boosting classifier with the Bagging and Random forests classifiers from the previous notebook (week #2).

### 📝 Excercise 2: Hyperparameter tuning 

**Challenge:** `Gradient boosted trees` need careful tuning of **`n_estimators`**, **`learning_rate`**, and **`loss`**.

**So, take the following steps:**

1. **Resplit** original train → 80 % train / 20 % validation.  
2. **Search** hyper-parameters on this pair, scoring only on the validation split.  
3. **Retrain** the best setting on the entire original training data.  
4. **Test** once, on the untouched test set.  


> NOTE: Never tune hyper-parameters on the test set, because peeking at it inflates apparent performance. Evaluate every candidate only on a separate validation split. After choosing the best configuration, run it once on the untouched test set.



In [7]:
# Split the training data into (new) training and evaluation sets
r_state = 0 # random_state
X_train_HP, X_evaluation, y_train_HP, y_evaluation = train_test_split(X_train, y_train,
                                                    test_size=0.2,
                                                    stratify=y_train,
                                                    random_state=r_state)

In [8]:
# find the best setting for the hyperparameters using training and evaluation sets

n_estim = [50, 100, 150, 200, 250]
lr = [ 10 ** x for x in range(-6,2)]
ls = ['log_loss']
gbc = GradientBoostingClassifier()

settings = []
for n_estimators in n_estim:
    for learning_rate in lr:
        for loss in ls:
            gbc_HP = GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate,
                                loss=loss, random_state=0)
            gbc_HP.fit(X_train_HP, y_train_HP)
            accuracy_eval = accuracy_score(y_evaluation, gbc_HP.predict(X_evaluation))
            settings.append((n_estimators, learning_rate, loss, accuracy_eval))
best_accuracy_eval =  max(settings, key=lambda x: x[-1])
print("Best settings according to accuracy on evaluation {}".format(best_accuracy_eval))

Best settings according to accuracy on evaluation (150, 0.01, 'log_loss', 0.9625)


In [9]:
# training the model using the optimal parameters on the original training set
# optimal parametrs: n_estimators=150, learning_rate=0.01, 'loss=log_loss'

gbc = GradientBoostingClassifier(n_estimators=150, learning_rate=0.01,
                                loss='log_loss', random_state=0)
gbc.fit(X_train, y_train)

GradientBoostingClassifier(learning_rate=0.01, n_estimators=150, random_state=0)

In [10]:
# evaluate the performance on test set
test_accuracy = accuracy_score(y_test, gbc.predict(X_test))

print('Accuracy of Gradient boosting classifier on test set after hyperparameter tuning: ', round(test_accuracy, 3))

Accuracy of Gradient boosting classifier on test set after hyperparameter tuning:  0.947


## ℹ️ Hyperparameter tuning using GridSearchCV
[`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) aids in finding the best combination of the model's hyperparameters automatically. You need to provide it with a set of parameters to experiment with, and it systematically explores each possible combination of them.


---



### 📝 Exercise 3: Finding optimal hyperparameters 
Repeat Exercise 2 and find the best combination of the hyperparameters for Gradient boosting classifier.

#### Part 1: Form the `param_grid` 

Form the parameters grid for the number of trees `n_estimators`, the learning rate `learning_rate`, and loss function `loss`. The parameters grid is a dictionary with parameters names (str) as keys and lists of parameter settings to try as values.

In [11]:
# form the param_grid
parameters_grid = {'n_estimators':[50, 100, 150, 200, 250],
                   'learning_rate':[ 10 ** x for x in range(-6,2)],
                   'loss':['log_loss']}

#### Part 2: Initiate the estimator

- Initiate the estimator (here: Gradient boosting classifier) using the default values for the parameters.

In [12]:
# Make an instance of the Gradient boosting classifier
gbc = GradientBoostingClassifier()

#### Part 3: Initiate GridSearchCV using the estimator and parm_grid
- Initiate GridSearch CV using Gradient boosting classifier as estimator and parameters grid.


In [13]:
# see above: from sklearn.model_selection import GridSearchCV
gscv  = GridSearchCV(gbc, parameters_grid, verbose=1) 

#### Part 4: Train the model
- Train the model and find the best hyperparameter combination.

In [14]:
gscv.fit(X_train, y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


GridSearchCV(estimator=GradientBoostingClassifier(),
             param_grid={'learning_rate': [1e-06, 1e-05, 0.0001, 0.001, 0.01,
                                           0.1, 1, 10],
                         'loss': ['log_loss'],
                         'n_estimators': [50, 100, 150, 200, 250]},
             verbose=1)

#### Part 5: Extract optimal hyperparameter set

1. Extract optimal hyperparameter set and train the model using this parameters 
2. Compute the accuracy of this model on the test set.

In [15]:
cv_results_df = pd.DataFrame(gscv.cv_results_)
sorted_df = cv_results_df.sort_values(by=['rank_test_score'], ascending=True)
display(sorted_df)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_learning_rate,param_loss,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
33,0.201354,0.033419,0.000975,0.000607,1,log_loss,200,"{'learning_rate': 1, 'loss': 'log_loss', 'n_es...",0.9625,0.9625,0.9500,0.962025,0.949367,0.957278,0.006207,1
31,0.162647,0.013340,0.000594,0.000485,1,log_loss,100,"{'learning_rate': 1, 'loss': 'log_loss', 'n_es...",0.9625,0.9625,0.9500,0.962025,0.949367,0.957278,0.006207,1
34,0.189184,0.021785,0.000399,0.000489,1,log_loss,250,"{'learning_rate': 1, 'loss': 'log_loss', 'n_es...",0.9625,0.9500,0.9500,0.949367,0.949367,0.952247,0.005134,3
32,0.174956,0.016452,0.000596,0.000487,1,log_loss,150,"{'learning_rate': 1, 'loss': 'log_loss', 'n_es...",0.9625,0.9500,0.9500,0.949367,0.949367,0.952247,0.005134,3
30,0.114820,0.018839,0.000396,0.000485,1,log_loss,50,"{'learning_rate': 1, 'loss': 'log_loss', 'n_es...",0.9500,0.9500,0.9500,0.949367,0.949367,0.949747,0.000310,5
27,0.354059,0.023861,0.000998,0.000632,0.1,log_loss,150,"{'learning_rate': 0.1, 'loss': 'log_loss', 'n_...",0.9625,0.9125,0.9500,0.949367,0.962025,0.947278,0.018278,6
26,0.246414,0.017361,0.000397,0.000486,0.1,log_loss,100,"{'learning_rate': 0.1, 'loss': 'log_loss', 'n_...",0.9625,0.9375,0.9250,0.949367,0.962025,0.947278,0.014460,6
29,0.611032,0.054058,0.000600,0.000490,0.1,log_loss,250,"{'learning_rate': 0.1, 'loss': 'log_loss', 'n_...",0.9625,0.9250,0.9500,0.949367,0.949367,0.947247,0.012200,8
28,0.515829,0.069398,0.000797,0.000398,0.1,log_loss,200,"{'learning_rate': 0.1, 'loss': 'log_loss', 'n_...",0.9625,0.9250,0.9500,0.949367,0.949367,0.947247,0.012200,8
25,0.114795,0.013854,0.000799,0.000400,0.1,log_loss,50,"{'learning_rate': 0.1, 'loss': 'log_loss', 'n_...",0.9625,0.9375,0.9125,0.936709,0.962025,0.942247,0.018652,10


In [17]:
print('Optimal hyperparameter set:' , sorted_df.loc[33,'params'])


Optimal hyperparameter set: {'learning_rate': 1, 'loss': 'log_loss', 'n_estimators': 200}


In [18]:
# Make an instance of the Gradient boosting classifier using optimal hyperparameters
gbc = GradientBoostingClassifier(learning_rate= 1 , loss= 'exponential', n_estimators= 300, random_state=0)

# Train the model on the data (train set)
gbc.fit(X_train, y_train)

# Predict the labels of new data (test set)

test_prediction = gbc.predict(X_test)  

test_accuracy = accuracy_score(y_test, test_prediction)

print('Accuracy of Gradient boosting classifier on test set: ', round(test_accuracy, 3))


Accuracy of Gradient boosting classifier on test set:  0.947


#### **💡 Thought question** : 
Are there any differences between the results of Exercise #2 and #3? if so, how can we explain this difference?